# End-to-End Data Science Pipeline Testing

Notebook ini dibuat untuk mensimulasikan dan menjelaskan setiap langkah dalam arsitektur *Machine Learning* yang digunakan pada sistem OneForMind (Dopmymind).

Sesuai dengan tugas *End-to-End Data Science Pipeline*, proses ini mencakup:
1. **Data Ingestion / Collection**
2. **Data Cleaning & Wrangling**
3. **Exploratory Data Analysis (EDA)**
4. **Model Building & Training**
5. **Model Tuning**
6. **Model Deployment & MLOps**

## 1. Data Ingestion / Collection
Mengumpulkan data mentah berupa profil akademik. Di sini kita membuat mock dataset sebagai representasi ekstraksi dokumen.

In [ ]:
import pandas as pd
import numpy as np

# Simulasi data dari ekstraksi PDF/Word
data = {
    'student_id': [1, 2, 3, 4, 5],
    'coursework_text': [
        'Developed a machine learning model using Python and Scikit-Learn to predict housing prices. Cleaned missing values and tuned hyperparameters.',
        'Created a responsive frontend web application using Vue.js and Tailwind CSS. Implemented state management and API integration.',
        'Designed a relational database schema using PostgreSQL. Wrote complex SQL queries for data extraction and reporting.',
        None, # Simulasi missing data
        'Deployed docker containers using Kubernetes on AWS. Configured CI/CD pipelines with GitHub Actions.'
    ],
    'target_archetype': ['Machine Learning Engineer', 'Frontend Developer', 'Data Engineer', None, 'DevOps Engineer']
}

df = pd.DataFrame(data)
print("Raw Data:")
display(df.head())

**Penjelasan:** Pada tahap ini kita melakukan *Data Collection*. Dalam sistem aslinya, data ini berasal dari modul OCR dan PDF ekstraksi (`pipeline.py -> extract_file_text()`).

## 2. Data Cleaning & Wrangling
Menangani missing values, duplikasi, dan pemformatan teks (lower-casing, penghapusan tanda baca).

In [ ]:
import re

# 1. Drop missing values
df_clean = df.dropna().copy()

# 2. Text Preprocessing (Wrangling)
def clean_text(text):
    text = text.lower() # lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text) # remove punctuation
    return text

df_clean['cleaned_text'] = df_clean['coursework_text'].apply(clean_text)

print("Cleaned Data:")
display(df_clean[['cleaned_text', 'target_archetype']])

**Penjelasan:** *Data Cleaning* sangat vital agar model NLP tidak memproses karakter *noise*. Baris yang kosong (`None`) dibuang, dan semua teks diubah menjadi huruf kecil.

## 3. Exploratory Data Analysis (EDA)
Menganalisis frekuensi kata (korelasi fitur) pada dataset.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

vectorizer = CountVectorizer(stop_words='english')
X_eda = vectorizer.fit_transform(df_clean['cleaned_text'])

# Visualisasi kata terbanyak
word_freq = dict(zip(vectorizer.get_feature_names_out(), X_eda.toarray().sum(axis=0)))
sorted_words = dict(sorted(word_freq.items(), key=lambda item: item[1], reverse=True))

plt.figure(figsize=(10, 5))
sns.barplot(x=list(sorted_words.keys())[:10], y=list(sorted_words.values())[:10])
plt.title('Top 10 Most Frequent Words in Coursework')
plt.xticks(rotation=45)
plt.show()

**Penjelasan:** *EDA* memvisualisasikan data untuk menemukan *pattern* atau kata kunci penting yang mendominasi dataset, seperti 'python', 'model', atau 'data'.

## 4. Model Building & Training
Membangun *Baseline Model* untuk melakukan klasifikasi Archetype menggunakan algoritma Naive Bayes atau SVM.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Feature Extraction menggunakan TF-IDF
tfidf = TfidfVectorizer(stop_words='english')

# Memisahkan Data
X_train, X_test, y_train, y_test = train_test_split(df_clean['cleaned_text'], df_clean['target_archetype'], test_size=0.2, random_state=42)

# Build Model Pipeline
model = make_pipeline(tfidf, MultinomialNB())

# Train Model
model.fit(X_train, y_train)

# Evaluasi Baseline Model
predictions = model.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))

**Penjelasan:** TF-IDF (*Term Frequency-Inverse Document Frequency*) digunakan mengubah teks menjadi vektor angka. Model klasifikasi (Naive Bayes) dilatih dengan data pelatihan (*fit*).

## 5. Model Tuning (Optimasi)
Mencari parameter model (*hyperparameter tuning*) yang paling optimal menggunakan GridSearchCV.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Parameter grid untuk diujicoba
param_grid = {
    'multinomialnb__alpha': [0.1, 0.5, 1.0, 2.0],
}

# Inisiasi GridSearchCV dengan cross-validation
grid_search = GridSearchCV(model, param_grid, cv=2)

# Karena dataset kita (mock data) sangat kecil untuk CV, kita tangkap error-nya agar tidak gagal di notebook ini.
try:
    grid_search.fit(X_train, y_train)
    print("Best Parameters:", grid_search.best_params_)
    best_model = grid_search.best_estimator_
except Exception as e:
    print("Tuning dilewati karena jumlah sampel terlalu sedikit untuk Cross Validation.")
    best_model = model

**Penjelasan:** Pada dataset besar, *Model Tuning* memastikan kita mendapat parameter terbaik (misal nilai alpha pada Naive Bayes) sehingga akurasi prediksi metrik meningkat maksimal.

## 6. Model Deployment & MLOps
Serialisasi (menyimpan) model agar bisa digunakan di Production Environment webapp (Laravel backend).

In [ ]:
import pickle

# Menyimpan model ke format .pkl
# File ini yang akan dibaca oleh `pipeline.py` di backend web!
with open('tfidf_vectorizer_test.pkl', 'wb') as f:
    pickle.dump(best_model.named_steps['tfidfvectorizer'], f)

with open('classifier_test.pkl', 'wb') as f:
    pickle.dump(best_model.named_steps['multinomialnb'], f)

print("Model Serialization complete! Files saved as .pkl for Production.")

# Simulasi pemanggilan model di backend (seperti yang dilakukan di pipeline.py)
sample_student_text = "I built an artificial neural network using tensorflow and python for classification."
test_vector = best_model.named_steps['tfidfvectorizer'].transform([sample_student_text])
test_prediction = best_model.named_steps['multinomialnb'].predict(test_vector)
test_probs = best_model.named_steps['multinomialnb'].predict_proba(test_vector)[0]

print(f"\nBackend Inference Test:")
print(f"Predicted Archetype: {test_prediction[0]}")
print("Probabilities per class:")
for cls, prob in zip(best_model.named_steps['multinomialnb'].classes_, test_probs):
    print(f" - {cls}: {prob*100:.2f}%")

**Penjelasan:** File `.pkl` yang dihasilkan (Serialization) dikirim/diunggah ke environment *Production*. Sistem backend (`ProcessCoursework.php` -> `pipeline.py`) kemudian akan membaca `.pkl` tersebut untuk memproses file coursework pengguna secara otomatis dalam bentuk pipeline (MLOps).